In [1]:
%%writefile tokenizer.py
import json


class Tokenizer:
  def __init__(self, vocab_file):
    with open(vocab_file, "r") as f:
      self.vocab = json.load(f)
      self.reverse_vocab = {v: k for k, v in self.vocab.items()}

  def encode(self, text):
    tokens = [] 
       
    for word in text.split():
      i = 0
      # example: states
      # state => 4
      # s => 58
      while i < len(word):
        found_match = False
        for j in range(len(word), i, -1):
          sub_word = word[i:j]
          if sub_word in self.vocab:
            tokens.append(self.vocab[sub_word])
            i = j
            found_match = True
            break
        if not found_match:
          tokens.append(self.vocab["<unk>"])
          i += 1
      tokens.append(self.vocab[" "])

    tokens.pop()
    return tokens
  


  def decode(self, ids):
    text = ""
    for id in ids:
      text += self.reverse_vocab[id]
    return text

Writing tokenizer.py


In [2]:
%%writefile tokenizer.json

{
  "the": 0,
  "capital": 1,
  "of": 2,
  "united": 3,
  "state": 4,
  "is": 5,
  "not": 6,
  "london": 7,
  "france": 8,
  "paris": 9,
  "and": 10,
  "berlin": 11,
  "germany": 12,
  "rome": 13,
  "in": 14,
  "italy": 15,
  "madrid": 16,
  "spain": 17,
  "lisbon": 18,
  "portugal": 19,
  "kingdom": 20,
  "washington": 21,
  "although": 22,
  "these": 23,
  "place": 24,
  "are": 25,
  "often": 26,
  "mention": 27,
  "together": 28,
  "each": 29,
  "country": 30,
  "has": 31,
  "its": 32,
  "own": 33,
  "identity": 34,
  "any": 35,
  "european": 36,
  "city": 37,
  "remain": 38,
  "important": 39,
  "with": 40,
  "a": 41,
  "rich": 42,
  "history": 43,
  "culture": 44,
  "europe": 45,
  "made": 46,
  "many": 47,
  "unique": 48,
  "world": 49,
  "while": 50,
  "known": 51,
  "for": 52,
  "art": 53,
  "fashion": 54,
  "famous": 55,
  "they": 56,
  "ed": 57,
  "s": 58,
  ".": 59,
  ",": 60,
  " ": 61,
  "<unk>": 62,
  "<pad>": 63
}

Writing tokenizer.json


In [3]:
%%writefile text.txt

the capital of the united states is not london. the capital of france is paris, and berlin is the capital of germany. rome is in italy, 

madrid is in spain, and lisbon is in portugal. the capital of the united kingdom is not paris, and the capital of the united states is not berlin. 
although these places are often mentioned together, although these capitals are often mentioned together, although these are often mentioned together, 
each country has its own capital, and each country has its own city, and each capital has its own identity, and each capital has its own history. washington 
is the capital of the united states, and london is the capital of the united kingdom. paris is known for art and fashion, and berlin is known for art and 
history, and rome is known for art and history, and madrid is known for culture and history, and lisbon is known for culture and art. rome is rich with culture, 
rome is rich with history, rome is rich with art, and madrid is rich with art and culture. lisbon is a unique city in portugal with a rich history, a rich culture, 
and a rich identity. these capitals are often mentioned together, these capitals are often mentioned together in art, these capitals are often mentioned together 
in culture, these capitals are often mentioned together in history. the united states is not in europe, the united states is not in any european place, and 
washington is not in any european city. each european country is made of important capitals, and each european capital is made of art, history, and culture. 
the capital of the united states is washington, the capital of the united kingdom is london, the capital of france is paris, the capital of germany is berlin, 
the capital of italy is rome, the capital of spain is madrid, and the capital of portugal is lisbon. while these capitals are in europe, while these capitals are 
in europe, washington is in the united states. these capitals remain important, these remain important, these places remain important in the world. the 
capital of each country is its own, the capital of each country is its identity, the capital of each country is its culture. europe is made of many, 
europe is made of many capitals, europe is made of many important places. each place is rich with culture, each place is rich with history, and each capital is 

rich with identity. the world is made of capitals, the world is made of, the world is made of places, and the capital of the united states is washington, 
not any european city, not paris, not london, not berlin. the capital of the united states is not london. the capital of france is paris, and berlin is the capital of germany.
rome is in italy, madrid is in spain, and lisbon is in portugal. the capital of the united kingdom is not paris, and the capital of the united 
states is not berlin. although these places are often mentioned together, each country has its own capital, and each capital has its own identity. 
washington is the capital of the united states, and london is the capital of the united kingdom. paris is known for art and fashion, while berlin is 
famous for its culture and history. rome is rich with history, and madrid is known for its art and culture. lisbon is a unique city in portugal 
with a rich history. these capitals are often mentioned together, although each place with its own culture. the united states is not in europe, 
and washington is not in any european country. these european capitals are made of history, culture, and identity. each country in europe has a capital, 
and each capital is known for important. london, paris, berlin, rome, madrid, and lisbon remain important places in the world. while these capitals
are in europe, washington is in the united states. although these places are not in the country, they are often mentioned together in art, culture, 
and history. the capital of each country is its own. europe is made of many capitals, and each has a capital with a unique history. 
the world is of important places, and the capital of the united states is washington, not any european city.


Writing text.txt


In [4]:
import torch
print("GPU Var mı?", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Aktif GPU:", torch.cuda.get_device_name(0))

GPU Var mı? True
Aktif GPU: Tesla T4


In [5]:
!pip install tiktoken transformers datasets -q

In [6]:
text1 = "The cat chased the dog"
text2 = "The dog chased the cat"

text = "The capital of france is"

In [7]:
vocab = {
    "The": 0,
    "cat": 1,
    "dog": 2,
    "chased": 3,
    "capital": 4,
    "of": 5,
    "France": 6,
    "is": 7,
    "<unk>": 8,
    "the": 9
}


def tokenize2(text):
    parts = text.split()
    ids = []
    for part in parts:
      if part in vocab:
        value = vocab[part]
      else:
        value = vocab["<unk>"]
      ids.append(value)
    return ids

token_ids1 = tokenize2(text1)
token_ids1

[0, 1, 3, 9, 2]

In [8]:
reverse_vocab = {id: part for part, id in vocab.items()}
reverse_vocab

{0: 'The',
 1: 'cat',
 2: 'dog',
 3: 'chased',
 4: 'capital',
 5: 'of',
 6: 'France',
 7: 'is',
 8: '<unk>',
 9: 'the'}

In [9]:
def detokenize(ids):
  text = ""
  for id in ids:
    part = reverse_vocab[id]
    text += part + " "
  text = text.strip()
  return text
  
detokenize(token_ids1)

'The cat chased the dog'

In [10]:
from huggingface_hub import login
import getpass

# Hücreyi çalıştırdığında üstte veya altta bir girdi kutusu açılır
hf_token = getpass.getpass("Hugging Face Token'ınızı yapıştırın: ")
login(token=hf_token)

In [11]:
#loading model directly
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained("google/gemma-3-27b-it")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/972 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [12]:
gemma_ids = processor.tokenizer.encode(text1)
gemma_ids

[2, 818, 5866, 83755, 506, 4799]

In [13]:
processor.tokenizer.decode(gemma_ids)
# 2 numaralı token gemma modelinde beginning of sequence ifade eder. her zaman

'<bos>The cat chased the dog'

In [14]:
from tokenizer import Tokenizer

tokenizer = Tokenizer("tokenizer.json")

tokenizer.encode("states"), tokenizer.decode([4, 58])

([4, 58], 'states')

In [15]:
with open("text.txt", "r") as f:
  text = f.read()

text

'\nthe capital of the united states is not london. the capital of france is paris, and berlin is the capital of germany. rome is in italy, \n\nmadrid is in spain, and lisbon is in portugal. the capital of the united kingdom is not paris, and the capital of the united states is not berlin. \nalthough these places are often mentioned together, although these capitals are often mentioned together, although these are often mentioned together, \neach country has its own capital, and each country has its own city, and each capital has its own identity, and each capital has its own history. washington \nis the capital of the united states, and london is the capital of the united kingdom. paris is known for art and fashion, and berlin is known for art and \nhistory, and rome is known for art and history, and madrid is known for culture and history, and lisbon is known for culture and art. rome is rich with culture, \nrome is rich with history, rome is rich with art, and madrid is rich with art

In [16]:
tokens = tokenizer.encode(text)
tokens

[0,
 61,
 1,
 61,
 2,
 61,
 0,
 61,
 3,
 61,
 4,
 58,
 61,
 5,
 61,
 6,
 61,
 7,
 59,
 61,
 0,
 61,
 1,
 61,
 2,
 61,
 8,
 61,
 5,
 61,
 9,
 60,
 61,
 10,
 61,
 11,
 61,
 5,
 61,
 0,
 61,
 1,
 61,
 2,
 61,
 12,
 59,
 61,
 13,
 61,
 5,
 61,
 14,
 61,
 15,
 60,
 61,
 16,
 61,
 5,
 61,
 14,
 61,
 17,
 60,
 61,
 10,
 61,
 18,
 61,
 5,
 61,
 14,
 61,
 19,
 59,
 61,
 0,
 61,
 1,
 61,
 2,
 61,
 0,
 61,
 3,
 61,
 20,
 61,
 5,
 61,
 6,
 61,
 9,
 60,
 61,
 10,
 61,
 0,
 61,
 1,
 61,
 2,
 61,
 0,
 61,
 3,
 61,
 4,
 58,
 61,
 5,
 61,
 6,
 61,
 11,
 59,
 61,
 22,
 61,
 23,
 61,
 24,
 58,
 61,
 25,
 61,
 26,
 61,
 27,
 57,
 61,
 28,
 60,
 61,
 22,
 61,
 23,
 61,
 1,
 58,
 61,
 25,
 61,
 26,
 61,
 27,
 57,
 61,
 28,
 60,
 61,
 22,
 61,
 23,
 61,
 25,
 61,
 26,
 61,
 27,
 57,
 61,
 28,
 60,
 61,
 29,
 61,
 30,
 61,
 31,
 61,
 32,
 61,
 33,
 61,
 1,
 60,
 61,
 10,
 61,
 29,
 61,
 30,
 61,
 31,
 61,
 32,
 61,
 33,
 61,
 37,
 60,
 61,
 10,
 61,
 29,
 61,
 1,
 61,
 31,
 61,
 32,
 61,
 33,
 61,
 34,
 60,
 

In [17]:
%pip install sentencepiece -qU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 93.3 MB/s eta 0:00:00


In [ ]:
import sentencepiece as spm

spm.SentencePieceTrainer.Train(
  input="text.txt",
  model_prefix="spm_tokenizer",
  vocab_size=64,
  model_type="bpe",
)

In [19]:
spm_tokenizer = spm.SentencePieceProcessor(model_file="spm_tokenizer.model")

spm_ids = spm_tokenizer.Encode(text1)
spm_tokens = spm_tokenizer.Encode(text1, out_type=str)
spm_ids, spm_tokens

([39, 0, 48, 43, 7, 41, 40, 7, 48, 41, 46, 30, 9, 39, 51, 45, 60],
 ['▁',
  'T',
  'h',
  'e',
  '▁c',
  'a',
  't',
  '▁c',
  'h',
  'a',
  's',
  'ed',
  '▁the',
  '▁',
  'd',
  'o',
  'g'])

In [20]:
%pip install tokenizers -qU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 132.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.23.3 which is incompatible.


In [21]:

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

In [22]:
hf_tokenizer = Tokenizer(BPE())

In [25]:
hf_tokenizer.pre_tokenizer = Whitespace()

In [24]:
trainer = BpeTrainer(vocab_size=64, special_tokens=["<unk>"])

hf_tokenizer.train(["text.txt"], trainer)

In [31]:
hf_tokenizer.get_vocab_size(), hf_tokenizer.encode(text1).ids, hf_tokenizer.decode([10, 7, 5, 3, 21, 5, 10, 49, 44, 28, 6, 16, 9])

(64,
 [10, 7, 5, 3, 21, 5, 10, 49, 44, 28, 6, 16, 9],
 'h e c a t c h as ed the d o g')

In [27]:
from transformers import PreTrainedTokenizerFast

fast_tokenizer = PreTrainedTokenizerFast(tokenizer_file="hf_tokenizer.json")

fast_tokenizer.encode(text1)

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.